In [3]:
"""
Step 3c of 5 — outcome-source sensitivity (Section V-A).

The MAEs in step 3b score all three institutions against one outcome series,
the April 2026 WEO. That series is the IMF's own, and the IMF is one of the
three institutions being scored, so a referee is entitled to ask whether the
accuracy ranking is an artefact of the scorer. This rescoresx everything against
the current AMECO vintage instead and reports how little moves.

UPLOAD   forecast_errors.csv   iso, round, target, IMF, OECD, EC, actual  (03a)
         AMECO6.TXT            AMECO current vintage, series OVGD
WRITES   rescore.csv           per-country MAE under both outcome series
"""
import os
import numpy as np
import pandas as pd

INST = ["IMF", "OECD", "EC"]
MIN_ROUNDS = 3
AMECO_ISO_FIX = {"ROM": "ROU"}

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def pick(tokens, exts, cols=None):
    for f in sorted(os.listdir(".")):
        low = f.lower()
        if not low.endswith(exts) or not any(t in low for t in tokens):
            continue
        if cols is None:
            return f
        try:
            if cols.issubset(pd.read_csv(f, nrows=1).columns):
                return f
        except Exception:
            pass
    return None


ERRCOLS = {"iso", "target", "actual", *INST}
NEEDED = [(("forecast", "error"), (".csv",), ERRCOLS, "forecast_errors.csv"),
          (("ameco",), (".txt",), None, "AMECO6.TXT")]


def missing():
    return [lab for tok, ext, c, lab in NEEDED if pick(tok, ext, c) is None]


gaps = missing()
for _ in range(4):
    if not gaps or not IN_COLAB:
        break
    print(f"still needed: {', '.join(gaps)}")
    files.upload()
    gaps = missing()
if gaps:
    raise FileNotFoundError("still missing: " + ", ".join(gaps) +
                            f"\nPresent: {sorted(os.listdir('.'))}")

F_ERR = pick(("forecast", "error"), (".csv",), ERRCOLS)
F_AMECO = pick(("ameco",), (".txt",))
print(f"errors  {F_ERR}\nAMECO   {F_AMECO}\n")

e = pd.read_csv(F_ERR)

# --- alternative outcomes: current AMECO vintage, growth from OVGD levels ---
am = pd.read_csv(F_AMECO, sep=";", encoding="latin-1", low_memory=False)
am.columns = [c.strip() for c in am.columns]
am["CODE"] = am.CODE.astype(str).str.strip()
ov = am[am.CODE.str.endswith(".OVGD")].copy()
ov["iso"] = ov.CODE.str.split(".").str[0].replace(AMECO_ISO_FIX)

alt = {}
for _, r in ov.iterrows():
    for y in sorted(e.target.unique()):
        try:
            alt[(r.iso, y)] = (float(r[str(y)]) / float(r[str(y - 1)]) - 1) * 100
        except Exception:
            pass
e["actual_ameco"] = [alt.get((i, t), np.nan) for i, t in zip(e.iso, e.target)]

miss = sorted(set(e.loc[e.actual_ameco.isna(), "iso"]))
if miss:
    print(f"no AMECO outcome for: {', '.join(miss)} (excluded from the comparison)")
e = e[e.actual_ameco.notna()]


def maes(outcome_col):
    d = e.copy()
    for k in INST:
        d["s_" + k] = d[k] - d[outcome_col]
    m = d.groupby("iso")[["s_" + k for k in INST]].agg(lambda s: s.abs().mean())
    m.columns = INST
    return m[d.groupby("iso").size() >= MIN_ROUNDS]


weo, ame = maes("actual"), maes("actual_ameco")
common = weo.index.intersection(ame.index)
weo, ame = weo.loc[common], ame.loc[common]

out = weo.add_suffix("_weo").join(ame.add_suffix("_ameco"))
out["best_weo"] = weo.idxmin(axis=1)
out["best_ameco"] = ame.idxmin(axis=1)
out["changed"] = out.best_weo != out.best_ameco
out.round(4).to_csv("rescore.csv")

# ---------------------------------------------------------------- report ----
chg = out.index[out.changed].tolist()
print(f"\n{len(common)} economies compared")
print(f"most accurate institution changes in {len(chg)}: "
      f"{', '.join(chg) if chg else 'none'}")
for i in chg:
    print(f"   {i}: WEO-scored {out.best_weo[i]} -> AMECO-scored {out.best_ameco[i]}")

print("\ncross-country MAE rank correlation, WEO-scored vs AMECO-scored")
print("(one coefficient per institution across the panel -- NOT a per-country")
print(" correlation, which with three institutions could only be +/-1 or +/-0.5)")
for k in INST:
    rho = weo[k].rank().corr(ame[k].rank(), method="pearson")
    print(f"   {k:<5}{rho:.4f}")

print("\npanel MAE")
print("   WEO-scored    " + "  ".join(f"{k} {weo[k].mean():.3f}" for k in INST))
print("   AMECO-scored  " + "  ".join(f"{k} {ame[k].mean():.3f}" for k in INST))

if IN_COLAB:
    files.download("rescore.csv")

errors  forecast_errors.csv
AMECO   AMECO6.TXT


36 economies compared
most accurate institution changes in 1: NZL
   NZL: WEO-scored EC -> AMECO-scored OECD

cross-country MAE rank correlation, WEO-scored vs AMECO-scored
(one coefficient per institution across the panel -- NOT a per-country
 correlation, which with three institutions could only be +/-1 or +/-0.5)
   IMF  0.9936
   OECD 0.9722
   EC   0.9997

panel MAE
   WEO-scored    IMF 1.804  OECD 1.911  EC 1.835
   AMECO-scored  IMF 1.805  OECD 1.891  EC 1.853


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>